In [1]:
import pandas as pd
import numpy as np

import os
from os.path import join
from pathlib import Path
from os import makedirs, remove
import getpass

import seaborn as sns
import matplotlib.pyplot as plt

import requests
import zipfile

In [4]:
username = getpass.getuser()

if username == "JOEL":
    main_path = Path("D:/Github/PE1333_Grupo8")

elif username == "USUARIO_alterno":
    main_path = Path("C:/Users/USUARIO_alterno/....")

print(f"Directorio del proyecto: {main_path}")

Directorio del proyecto: D:\Github\PE1333_Grupo8


In [5]:
# ============================================================
# 1. DEFINIR DIRECTORIOS PRINCIPALES
# ============================================================
tp = 1
a_path = join(main_path, "TP_0"+str(tp), "A_Inputs")
b_path = join(main_path, "TP_0"+str(tp), "B_Outputs")
c_path = join(main_path, "TP_0"+str(tp), "C_Programs")
d_path = join(main_path, "TP_0"+str(tp), "D_LaTeX")

# ============================================================
# 2. DEFINIR DIRECTORIOS POR TRABAJO PRÁCTICO
# ============================================================

b_path_data = join(b_path, "B_01_Data")
b_path_figr = join(b_path, "B_01_Figure")
b_path_tabl = join(b_path, "B_01_Table")
b_path_text = join(b_path, "B_01_Text")


In [6]:
for year in [2026]:
    for trim in range(1, 2):

        url = (
            "https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph/"
            f"EPH_usu_{trim}_Trim_{year}_txt.zip"
        )

        destino = a_path
        makedirs(destino, exist_ok=True)

        response = requests.get(url, timeout=120)

        if response.status_code != 200:
            print(f"No disponible: {year}-T{trim}")
            continue

        archivo_zip = join(destino, "EPH.zip")

        with open(archivo_zip, "wb") as f:
            f.write(response.content)

        try:
            with zipfile.ZipFile(archivo_zip, "r") as z:
                z.extractall(destino)

            remove(archivo_zip)
            print(f"Descargado: {year}-T{trim}")

        except zipfile.BadZipFile:
            print(f"Archivo ZIP inválido: {year}-T{trim}")

Descargado: 2026-T1


In [7]:
datos_indiv_raw = pd.read_csv(join(a_path, "usu_individual_T126.txt"), sep=";", encoding="utf-8")
datos_hogar_raw = pd.read_csv(join(a_path, "usu_hogar_T126.txt"), sep=";", encoding="utf-8")


C:\Users\JOEL\AppData\Local\Temp\ipykernel_21608\2081912851.py:1: DtypeWarning: Columns (0: PP09A_ESP) have mixed types. Specify dtype option on import or set low_memory=False.
  datos_indiv_raw = pd.read_csv(join(a_path, "usu_individual_T126.txt"), sep=";", encoding="utf-8")


In [55]:
# OCUPADOS
datos_indiv = datos_indiv_raw.loc[datos_indiv_raw["ESTADO"] == 1].copy()
datos_indiv = datos_indiv.loc[datos_indiv["EMPLEO"].isin([1, 2])]

# VARIABLE INFORMAL
datos_indiv["informal"] = (datos_indiv["EMPLEO"].map({1: 0, 2: 1}).astype(int))

In [56]:
# ============================================================
# 1.5. SELECCIÓN DE VARIABLES SOCIOECONÓMICAS
# ============================================================

# Características demográficas
variables_demograficas = [
    "CH03",       # Relación de parentesco
    "CH04",       # Sexo
    "CH06",       # Edad
    "CH07",       # Estado conyugal
    "CH15",       # Lugar de nacimiento
    "CH16"        # Lugar de residencia hace 5 años
]

# Capital humano
variables_educacion = [
    "CH09",       # Alfabetización
    "CH10",       # Asistencia educativa
    "NIVEL_ED"    # Nivel educativo
]

# Características laborales
variables_laborales = [
    "CAT_OCUP",   # Categoría ocupacional
    "PP03G",      # Deseo de trabajar más horas
    "INTENSI",    # Intensidad laboral
    "PP04A"       # Sector de gestión
]

# Características geográficas
variables_geograficas = [
    "REGION",     # Región
    "MAS_500"     # Tamaño del aglomerado
]

# Lista completa de predictoras
variables_x = (
    variables_demograficas
    + variables_educacion
    + variables_laborales
    + variables_geograficas
)

print("\nNúmero de variables seleccionadas:", len(variables_x))


Número de variables seleccionadas: 15


In [57]:
for var in variables_x:
    print(f"{var}: {np.sort(datos_indiv[var].unique())}")

CH03: [ 1  2  3  4  5  6  7  8  9 10]
CH04: [1 2]
CH06: [10 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34
 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58
 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82
 83 84 85 86 87 88 99]
CH07: [1 2 3 4 5]
CH15: [1 2 3 4 5 9]
CH16: [1 2 3 4 5 9]
CH09: [1 2 9]
CH10: [1 2 3]
NIVEL_ED: [1 2 3 4 5 6 7]
CAT_OCUP: [1 2 3 4]
PP03G: [1. 2. 9.]
INTENSI: [1. 2. 3. 4.]
PP04A: [1. 2. 3.]
REGION: [ 1 40 41 42 43 44]
MAS_500: ['N' 'S']


In [58]:
variables_ns_nr = ["CH15", "CH16", "CH09", "PP03G"]

datos_indiv[variables_ns_nr] = datos_indiv[variables_ns_nr].replace(9, np.nan)

In [59]:
for var in variables_x:
    print(f"{var}: {np.sort(datos_indiv[var].unique())}")

CH03: [ 1  2  3  4  5  6  7  8  9 10]
CH04: [1 2]
CH06: [10 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34
 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58
 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82
 83 84 85 86 87 88 99]
CH07: [1 2 3 4 5]
CH15: [ 1.  2.  3.  4.  5. nan]
CH16: [ 1.  2.  3.  4.  5. nan]
CH09: [ 1.  2. nan]
CH10: [1 2 3]
NIVEL_ED: [1 2 3 4 5 6 7]
CAT_OCUP: [1 2 3 4]
PP03G: [ 1.  2. nan]
INTENSI: [1. 2. 3. 4.]
PP04A: [1. 2. 3.]
REGION: [ 1 40 41 42 43 44]
MAS_500: ['N' 'S']


In [ ]:



# ============================================================
# 1.3. EXCLUIR OBSERVACIONES CON EMPLEO = 9
# ============================================================

# EMPLEO = 1: formal
# EMPLEO = 2: informal
# EMPLEO = 9: sin clasificación válida




# ============================================================
# 1.4. CONSTRUIR VARIABLE DEPENDIENTE
# ============================================================

datos_indiv["informal"] = (
    datos_indiv["EMPLEO"]
    .map({
        1: 0,
        2: 1
    })
    .astype(int)
)

print("\nDistribución de informal:")
print(
    datos_indiv["informal"]
    .value_counts()
    .sort_index()
)

print("\nDistribución porcentual:")
print(
    datos_indiv["informal"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nObservaciones válidas:", len(datos_indiv))




# Verificar que existen en la base
assert all(
    var in datos_indiv.columns
    for var in variables_x
)


# ============================================================
# 1.6. EXCLUIR VARIABLES QUE GENERAN DATA LEAKAGE
# ============================================================

variables_prohibidas = [
    "EMPLEO",
    "SECTOR",
    "PP07H",
    "PP07I",
    "PP07G1",
    "PP07G2",
    "PP07G3",
    "PP07G4",
    "PP07G_59",
    "PP07K",
    "PP07L",
    "PP05I",
    "PP05J",
    "PP05K"
]

assert not set(variables_x).intersection(
    variables_prohibidas
)

print("\nVerificación de data leakage: correcta")


# ============================================================
# 1.7. CREAR BASE DE TRABAJO
# ============================================================

# Los identificadores se conservan para futuros merges,
# pero NO forman parte de la matriz de predictores X.

variables_id = ["CODUSU", "NRO_HOGAR", "COMPONENTE"]
datos_tp = datos_indiv[variables_id + variables_x + ["informal"]].copy()
print("\nDimensiones de la base seleccionada:", datos_tp.shape)


# ============================================================
# 1.8. LIMPIEZA DE VALORES INVÁLIDOS
# ============================================================

# Los códigos especiales se tratan por variable.
# No reemplazamos todos los 9 de la base, porque
# algunas variables tienen el código 9 como válido.

codigos_validos = {
    "CH03": list(range(1, 11)),
    "CH04": [1, 2],
    "CH07": [1, 2, 3, 4, 5],
    "CH09": [1, 2],
    "CH10": [1, 2, 3],
    "CH15": [1, 2, 3, 4, 5],
    "CH16": [1, 2, 3, 4, 5],
    "NIVEL_ED": [1, 2, 3, 4, 5, 6, 7],
    "CAT_OCUP": [1, 2, 3, 4],
    "REGION": [1, 40, 41, 42, 43, 44],
    "MAS_500": ["N", "S"],
    "PP03G": [1, 2],
    "INTENSI": [1, 2, 3, 4],
    "PP04A": [1, 2, 3]
}

# Convertir variables categóricas numéricas
for var in codigos_validos:

    if var != "MAS_500":
        datos_tp[var] = pd.to_numeric(
            datos_tp[var],
            errors="coerce"
        )

    datos_tp[var] = datos_tp[var].where(
        datos_tp[var].isin(codigos_validos[var]),
        np.nan
    )

# Limpieza específica de edad
datos_tp["CH06"] = pd.to_numeric(
    datos_tp["CH06"],
    errors="coerce"
)

datos_tp["CH06"] = datos_tp["CH06"].where(
    datos_tp["CH06"].between(0, 98),
    np.nan
)


# ============================================================
# 1.9. ANALIZAR VALORES FALTANTES
# ============================================================

tabla_missing = pd.DataFrame({
    "N_faltantes": datos_tp[variables_x].isna().sum(),
    "Porcentaje": (
        datos_tp[variables_x].isna().mean() * 100
    ).round(2)
})

tabla_missing = tabla_missing.sort_values(
    "Porcentaje",
    ascending=False
)

print("\nValores faltantes por variable:")
display(tabla_missing)


# ============================================================
# 1.10. VERIFICACIONES FINALES
# ============================================================

# La variable dependiente debe ser binaria
assert datos_tp["informal"].isin([0, 1]).all()

# Verificar identificadores individuales únicos
assert not datos_tp.duplicated(
    subset=variables_id
).any()

# Verificar cantidad de observaciones
assert len(datos_tp) == 19560

# Comprobar que no se incluyeron variables prohibidas
assert not set(variables_x).intersection(
    variables_prohibidas
)

print("\n======================================")
print("RESUMEN FINAL - CONSIGNA 1")
print("======================================")
print("Observaciones:", len(datos_tp))
print("Predictores:", len(variables_x))
print("Formales:", (datos_tp["informal"] == 0).sum())
print("Informales:", datos_tp["informal"].sum())
print("Valores faltantes en X:",
      datos_tp[variables_x].isna().sum().sum())
print("======================================")

# Visualizar primeras observaciones
display(datos_tp.head())


Dimensiones originales: (43739, 234)
Cantidad de variables: 234

Distribución de ESTADO:
ESTADO
0       86
1    19573
2     1423
3    17983
4     4674
Name: count, dtype: int64

Distribución de EMPLEO:
EMPLEO
1.0    10762
2.0     8798
9.0       13
NaN    24166
Name: count, dtype: int64

Cruce entre ESTADO y EMPLEO:
EMPLEO    1.0   2.0  9.0    NaN
ESTADO                         
0           0     0    0     86
1       10762  8798   13      0
2           0     0    0   1423
3           0     0    0  17983
4           0     0    0   4674

Ocupados originales: 19573

Distribución de informal:
informal
0    10762
1     8798
Name: count, dtype: int64

Distribución porcentual:
informal
0    55.02
1    44.98
Name: proportion, dtype: float64

Observaciones válidas: 19560

Número de variables seleccionadas: 15

Verificación de data leakage: correcta

Dimensiones de la base seleccionada: (19560, 19)

Valores faltantes por variable:


,N_faltantes,Porcentaje
PP03G,6,0.03
CH16,1,0.01
CH06,1,0.01
CH09,1,0.01
CH15,2,0.01
CH04,0,0.00
CH03,0,0.00
CH07,0,0.00
CH10,0,0.00
NIVEL_ED,0,0.00



RESUMEN FINAL - CONSIGNA 1
Observaciones: 19560
Predictores: 15
Formales: 10762
Informales: 8798
Valores faltantes en X: 11


,CODUSU,NRO_HOGAR,COMPONENTE,CH03,CH04,CH06,CH07,CH15,CH16,CH09,CH10,NIVEL_ED,CAT_OCUP,PP03G,INTENSI,PP04A,REGION,MAS_500,informal
0,TQRMNORURHLLKUCDEOJAH00893620,1,2,2,2,44.0,2,2.0,1.0,1.0,2,6,3,2.0,2.0,1.0,44,N,0
5,TQRMNOQUPHLOLPCDEHLEH00893322,1,1,1,2,56.0,5,1.0,1.0,1.0,2,5,3,2.0,2.0,1.0,40,N,0
8,TQRMNOPTVHJMKTCDEOJAH00875741,1,1,1,1,52.0,3,2.0,1.0,1.0,2,4,2,2.0,2.0,2.0,44,N,0
9,TQRMNORVRHJKKMCDEHLEH00875561,1,1,1,2,48.0,3,1.0,1.0,1.0,2,2,3,2.0,2.0,2.0,40,N,1
10,TQRMNORVRHJKKMCDEHLEH00875561,1,2,3,2,26.0,5,1.0,1.0,1.0,2,4,3,2.0,2.0,2.0,40,N,0
